In [1]:
import os

from IPython.display import FileLink

if os.getcwd() == '/notebooks':
    os.chdir("./motion-synthesis")
    print('inside dir: ', os.listdir())
print(os.listdir())
print("Click here to download the dit_d0.tar: ", display(FileLink("/notebooks/motion-synthesis/dit_d0.tar")))


os.environ["TOKENIZERS_PARALLELISM"] = "false"

inside dir:  ['requirements.txt', 'networks', 'glove', 'exp_results', 'README.md', 'conda_requirements.txt', 'prepare', 'data_utils', 'dataset_split_builder.py', 'demo.py', 'checkpoints', 'environment.yaml', 'eval_models', 'environment_cpu.yaml', '.git', 'assets', 'options', 'main.ipynb', 'motion-dataset.zip', 'common', 'main.py', '.ipynb_checkpoints', 'data', '.gitignore', 'log', 'utils', 'dit_crossattn_debug.tar']
['requirements.txt', 'networks', 'glove', 'exp_results', 'README.md', 'conda_requirements.txt', 'prepare', 'data_utils', 'dataset_split_builder.py', 'demo.py', 'checkpoints', 'environment.yaml', 'eval_models', 'environment_cpu.yaml', '.git', 'assets', 'options', 'main.ipynb', 'motion-dataset.zip', 'common', 'main.py', '.ipynb_checkpoints', 'data', '.gitignore', 'log', 'utils', 'dit_crossattn_debug.tar']


/notebooks/motion-synthesis/dit_d0.tar

Click here to download the dit_d0.tar:  None


In [2]:
import torch
from torch.backends import cuda
from options.train_options  import TrainOptions
from os.path import join as pjoin
import os
from utils.paramUtils import t2m_kinematic_chain
import numpy as np
from utils.word_vectorizer import WordVectorizer
from torch.utils.data import DataLoader
from data_utils.dataset import MotionDatasetV2
from data_utils.dataset import PartMotionDatasetV2
from networks.nn import MotionVQVAE, DiT
from networks.trainers import MotionVQVAETrainer, MotionDiTTrainer
from torch.utils.data import Subset
from networks.nn_validator import VQVAEValidator, DiffusionValidator
import json

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7ff88e675010>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py", line 770, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [ ]:
parser = TrainOptions()
options = parser.parse(args = ['--max_epoch', '250', '--lr', '3e-4', '--save_latest', '50', '--eval_every_e', '5', '--save_every_e', '50', '--log_every', '5'])
options.gpu_id = torch.cuda.current_device() if torch.cuda.is_available() else -1
options.device = torch.device("cpu" if options.gpu_id==-1 else "cuda:" + str(options.gpu_id))
torch.autograd.set_detect_anomaly(True)

# disabling flash backend till the architecture parameters allow stable use of flash backend for attention
# without creating nans
cuda.enable_flash_sdp(False)
cuda.enable_mem_efficient_sdp(False)
cuda.enable_math_sdp(True)

if options.gpu_id != -1:
    # self.opt.gpu_id = int(self.opt.gpu_id)
    torch.cuda.set_device(options.gpu_id)

print('\nDevice used: ', options.device)
options.save_root = pjoin(options.checkpoints_dir, 'HumanML3D', options.name)
options.model_dir = pjoin(options.checkpoints_dir, 'model')
options.meta_dir = pjoin(options.save_root, 'meta')
options.eval_dir = pjoin(options.save_root, 'animation')
options.log_dir = pjoin('./log', options.dataset_name, options.name)
options.experiment_dir = './exp_results/onestep-diffusion-setup/finetune_dit_cross_attn'
options.output_dir = options.experiment_dir
#options.save_latest = 1
options.is_train = True
options.is_continue = False
options.dataset_mode = "debug"
options.batch_size = 64
options.model_filename = 'dit_crossattn_full.tar'

os.makedirs(options.model_dir, exist_ok=True)
os.makedirs(options.meta_dir, exist_ok=True)
os.makedirs(options.eval_dir, exist_ok=True)
os.makedirs(options.log_dir, exist_ok=True)

options.data_root = './data/HumanML3D'
options.motion_dir = pjoin(options.data_root, 'new_joint_vecs')
options.text_dir = pjoin(options.data_root, 'texts')
options.joints_num = 22
options.max_motion_length = 196
dim_pose = 263
radius = 4
fps = 20
kinematic_chain = t2m_kinematic_chain

print('Set parameters')
print('Learning rate: ', options.lr)
print('Max epochs: ', options.max_epoch)
print('Save latest frequency: ', options.save_latest)
print('Save every epoch frequency: ', options.save_every_e)
print('Log every iterations frequency: ', options.log_every)


Device used:  cuda:0
Set parameters
Learning rate:  0.0002
Max epochs:  250
Save latest frequency:  50
Save every epoch frequency:  50
Log every iterations frequency:  5


In [ ]:
mean = np.load(pjoin(options.data_root, 'Mean.npy'))
std = np.load(pjoin(options.data_root, 'Std.npy'))

w_vectorizer = WordVectorizer('./glove', 'our_vab')
train_split_fn = 'train.txt'
val_split_fn = 'val.txt'

if options.dataset_mode == "debug":
    train_split_fn = 'train_debug.txt'
    val_split_fn = 'val_debug.txt'
elif options.dataset_mode == "micro":
    train_split_fn = 'train_micro.txt'
    val_split_fn = 'val_micro.txt'

train_split_file = pjoin(options.data_root, train_split_fn)
val_split_file = pjoin(options.data_root, val_split_fn)

if options.dataset_mode in ["nano", "micro"]:
    train_dlen = 150 if options.dataset_mode == "micro" else 30
    val_dlen = 50 if options.dataset_mode == "micro" else 15

    subset_path = os.path.join(options.meta_dir, "micro_subsets.json")
    par_train_dataset = PartMotionDatasetV2(options, mean, std, train_split_file)
    par_val_dataset = PartMotionDatasetV2(options, mean, std, val_split_file)

    if os.path.exists(subset_path):
        with open(subset_path, "r") as f:
            subsets = json.load(f)
        micro_train_indices = subsets["micro_train_indices"]
        micro_val_indices = subsets["micro_val_indices"]
    else:
        seed = 42
        rng = np.random.default_rng(seed)

        all_train_indices = np.arange(len(par_train_dataset))
        all_val_indices = np.arange(len(par_val_dataset))

        micro_train_indices = rng.permutation(all_train_indices)[:train_dlen].tolist()
        micro_val_indices = rng.permutation(all_val_indices)[:val_dlen].tolist()

        with open(subset_path, "w") as f:
            json.dump({
                "seed": seed,
                "micro_train_indices": micro_train_indices,
                "micro_val_indices": micro_val_indices,
            }, f, indent=4)
    train_dataset = Subset(par_train_dataset, micro_train_indices)
    val_dataset = Subset(par_val_dataset, micro_val_indices)
else:
    train_dataset = PartMotionDatasetV2(options, mean, std, train_split_file)
    val_dataset = PartMotionDatasetV2(options, mean, std, val_split_file)

print('\nTotal number of snippets in train: ', len(train_dataset))
print('Total number of snippets in val: ', len(val_dataset))
sample_motion = train_dataset[8]
print('Sample data shape: ', sample_motion['motion_parts'].shape, sample_motion['text'])
Dp_max = sample_motion['motion_parts'].shape[-1]

id list 64


100%|██████████| 64/64 [00:00<00:00, 5046.25it/s]


Motion shape (B, T, D): (59, 199, 263)
Total number of motions 59, snippets 6440
id list 32


100%|██████████| 32/32 [00:00<00:00, 4365.23it/s]

Motion shape (B, T, D): (31, 170, 263)
Total number of motions 31, snippets 3188

Total number of snippets in train:  6440
Total number of snippets in val:  3188
Sample data shape:  (40, 6, 60) 


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                              shuffle=False, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                        shuffle=False, pin_memory=True)

if options.stage == "autoencoder":
    vqvae = MotionVQVAE(
        input_dim=Dp_max,
        enc_hidden_dim=1024,
        dec_hidden_dim=1024,
        latent_dim=256,
        num_embeddings=512,
        beta=0.1
    )

    if options.is_train:
        trainer = MotionVQVAETrainer(options, vqvae = vqvae)
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader)
else:
    dit = DiT(
        input_size = 512,
        hidden_size = 1152,
        text_dim = 384
    )

    if options.is_train:
        trainer = MotionDiTTrainer(
            args = options,
            dit = dit,
            autoencoder_type="pretrained_vae"
        )
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader
        )


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Number of epochs: 250
Number of steps: 25000
Iters Per Epoch, Training: 0100, Validation: 049

Epoch: 0
Train Loss: 3.62786
Validation Loss: 3.10735
Epoch: 10
Train Loss: 2.18156
Validation Loss: 2.25167
Epoch: 15
Train Loss: 2.03393
Validation Loss: 2.10142
Epoch: 20
Train Loss: 1.93168
Validation Loss: 2.02498
Epoch: 25
Train Loss: 1.84427
Validation Loss: 1.98297
Epoch: 30
Train Loss: 1.75850
Validation Loss: 1.97425
Epoch: 35
Train Loss: 1.72867
Validation Loss: 1.96863
Epoch: 40
Train Loss: 1.64759
Validation Loss: 1.93886
Epoch: 45
Train Loss: 1.60816
Validation Loss: 1.90931
Epoch: 50
Train Loss: 1.57289
Validation Loss: 1.91311


In [ ]:
if options.stage == "autoencoder":
    test_model_filepath = pjoin(options.model_dir, options.model_filename)

    if not(options.is_train) and os.path.exists(test_model_filepath):
        vqvae_model_dict = torch.load(test_model_filepath, map_location = options.device)

        vqvae.load_state_dict(vqvae_model_dict['vqvae'])

        vqvae_validator = VQVAEValidator(
            opt = options,
            vqvae=vqvae,
            train_dataloader=train_loader,
            val_dataloader = val_loader
        )
        vqvae_validator.validate()
    else:
        print("Invalid mode or model file doesn't exist!")
else:
    test_model_filepath = pjoin(options.model_dir, options.model_filename)

    if not(options.is_train) and os.path.exists(test_model_filepath):
        dit_model_dict = torch.load(test_model_filepath, map_location = options.device)

        dit.load_state_dict(dit_model_dict['dit'])

        dit_validator = DiffusionValidator(
            opt = options,
            dit=dit,
            train_dataloader=train_loader,
            val_dataloader = val_loader
        )
        dit_validator.validate()
    else:
        print("Invalid mode or model file doesn't exist!")

Invalid mode or model file doesn't exist!
